# Lecture 3: Control Your Agent with Hooks (hands-on)
**Strands Agents with Ollama · Lecture 3 of 5** · companion to `Lecture03_Hooks.pdf`

Hooks fire **every time** at lifecycle events, whatever the model decides. You will build:
1. An audit log of every tool call
2. A per-request rate limiter
3. A business-rule validator for refunds
4. Human-in-the-loop approval with `event.interrupt()`

## 0. Setup: Ollama + Strands
Same steps as `StrandsAgent_SetUp.ipynb`. Run these once per Colab session (about 2 to 3 minutes).

> **Tip:** *Runtime → Change runtime type → T4 GPU* makes Ollama much faster. CPU works too, just slower.

In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq zstd
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the Ollama server in the background and wait until it answers
import subprocess, time, urllib.request
subprocess.Popen("nohup ollama serve > /tmp/ollama.log 2>&1 &", shell=True)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start - check /tmp/ollama.log")

In [ ]:
!ollama pull qwen3:0.6b
# Optional, more reliable for tool use (1.4 GB):
# !ollama pull qwen3:1.7b

In [ ]:
!pip install -q "strands-agents[ollama,web-fetch]"

### Shared model helper
Every lecture uses the same `make_model()` helper. Change `MODEL_ID` here to switch models for the whole notebook.

In [ ]:
from strands.models.ollama import OllamaModel

MODEL_ID = "qwen3:0.6b"          # try "qwen3:1.7b" for better tool use

def make_model(model_id=MODEL_ID, **kwargs):
    return OllamaModel(
        host="http://127.0.0.1:11434",
        model_id=model_id,
        options={"num_ctx": 8192},   # explicit context window (see Lecture 4)
        **kwargs,
    )

import importlib.metadata as md_
print("strands-agents", md_.version("strands-agents"), "| model:", MODEL_ID)

## 1. Tools for this lab (all simulated, safe to run)

In [ ]:
from strands import Agent, tool

@tool
def calculator(a: float, b: float, operation: str = "add") -> str:
    """Perform a math operation on two numbers.

    Args:
        a: First number
        b: Second number
        operation: One of "add", "subtract", "multiply", "divide", "power"
    """
    ops = {"add": a + b, "subtract": a - b, "multiply": a * b,
           "divide": a / b if b else "Error: division by zero", "power": a ** b}
    return str(ops.get(operation, f"Unknown operation: {operation}"))

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: City name, e.g. "Hanoi"
    """
    fake = {"hanoi": "31°C, humid", "tokyo": "18°C, clear", "paris": "12°C, rain",
            "london": "10°C, cloudy", "sydney": "24°C, sunny"}
    return fake.get(city.lower(), "20°C, unknown conditions")

@tool
def process_refund(order_id: str, amount: float) -> str:
    """Issue a refund to a customer for an order.

    Args:
        order_id: The order ID, e.g. "A-1042"
        amount: Refund amount in US dollars
    """
    return f"Refunded ${amount:.2f} for order {order_id}"

@tool
def delete_file(path: str) -> str:
    """Delete a file at the given path.

    Args:
        path: Path of the file to delete
    """
    return f"Deleted {path}"     # simulated - nothing is really deleted

In [ ]:
from strands.hooks import (HookProvider, HookRegistry, BeforeInvocationEvent,
                           BeforeToolCallEvent, AfterToolCallEvent,
                           BeforeModelCallEvent, AfterModelCallEvent)
import strands.hooks
print([n for n in dir(strands.hooks) if n.endswith("Event")])

## 2. Audit log hook

In [ ]:
import time

class AuditLogHook(HookProvider):
    def __init__(self):
        self.log, self._start = [], {}

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.before)
        registry.add_callback(AfterToolCallEvent, self.after)

    def before(self, event: BeforeToolCallEvent) -> None:
        self._start[event.tool_use["toolUseId"]] = time.time()

    def after(self, event: AfterToolCallEvent) -> None:
        tid = event.tool_use["toolUseId"]
        self.log.append({"tool": event.tool_use["name"],
                         "input": event.tool_use["input"],
                         "status": event.result["status"],
                         "seconds": round(time.time() - self._start.pop(tid, time.time()), 3)})

audit = AuditLogHook()
agent = Agent(model=make_model(), tools=[calculator, get_weather], hooks=[audit], callback_handler=None)
print(agent("What is 12 times 7? Use the calculator."))
for row in audit.log:
    print(row)

## 3. Rate limiter
The counter resets on `BeforeInvocationEvent`, so the limit is **per request**. `cancel_tool` skips the call and tells the model why.

In [ ]:
class LimitToolCounts(HookProvider):
    def __init__(self, max_calls: int = 3):
        self.max_calls = max_calls
        self.counts: dict[str, int] = {}

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset)
        registry.add_callback(BeforeToolCallEvent, self.check)

    def reset(self, event: BeforeInvocationEvent) -> None:
        self.counts = {}

    def check(self, event: BeforeToolCallEvent) -> None:
        name = event.tool_use["name"]
        self.counts[name] = self.counts.get(name, 0) + 1
        if self.counts[name] > self.max_calls:
            event.cancel_tool = (f"'{name}' hit the {self.max_calls}-call limit. "
                                 "Do NOT call this tool again.")

agent = Agent(model=make_model(), tools=[get_weather], hooks=[LimitToolCounts(max_calls=2)],
              system_prompt="Use get_weather once per city.")
result = agent("Get the weather for Hanoi, Tokyo, Paris, London and Sydney.")

In [ ]:
# What did the model see when it was blocked?
for m in agent.messages:
    for b in m["content"]:
        if "toolResult" in b:
            print(b["toolResult"]["status"], "|", str(b["toolResult"]["content"])[:100])

## 4. Business-rule validation: refund policy

In [ ]:
class RefundPolicyHook(HookProvider):
    def __init__(self, max_amount: float = 100.0):
        self.max_amount = max_amount

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.check)

    def check(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use["name"] != "process_refund":
            return
        amount = float(event.tool_use["input"].get("amount", 0))
        if amount > self.max_amount:
            event.cancel_tool = (f"Refund of ${amount} exceeds the ${self.max_amount} policy limit. "
                                 "Tell the customer a manager will review it.")

support = Agent(model=make_model(), tools=[process_refund], hooks=[RefundPolicyHook(100)],
                system_prompt="You are a support agent. Use process_refund to issue refunds.")
support("Please refund $50 for order A-1042.")
print("\n" + "-" * 40)
support("Please refund $500 for order B-2077.")

## 5. Human-in-the-loop approval
`event.interrupt()` pauses the run and returns control to you with `stop_reason == "interrupt"`. You resume by calling the agent with `interruptResponse` blocks.

In [ ]:
class DeleteApprovalHook(HookProvider):
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.check_delete)

    def check_delete(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use["name"] != "delete_file":
            return
        approval = event.interrupt("delete-approval",
                                   reason={"path": event.tool_use["input"].get("path")})
        if str(approval).lower() != "y":
            event.cancel_tool = "User denied file deletion"

agent = Agent(model=make_model(), tools=[delete_file], hooks=[DeleteApprovalHook()],
              system_prompt="You manage files. Use delete_file when asked.")

result = agent("Delete the file old_report.txt")

while result.stop_reason == "interrupt":
    responses = []
    for interrupt in result.interrupts:
        print(f"\nApproval needed: {interrupt.name} -> {interrupt.reason}")
        answer = input("Approve? (y/n): ")
        responses.append({"interruptResponse": {"interruptId": interrupt.id, "response": answer}})
    result = agent(responses)

print("\nFINAL:", result)

## 6. Your turn
**Exercise 1:** Run section 5 again and answer **n**. What does the model tell the user?

**Exercise 2:** Stack `AuditLogHook`, `LimitToolCounts` and `RefundPolicyHook` on **one** agent with all four tools and check they all work together.

**Exercise 3 (stretch):** Write a `TimingHook` using `BeforeModelCallEvent` / `AfterModelCallEvent` that prints how long each Ollama call takes. Compare `think=True` vs `think=False`.

In [ ]:
class TimingHook(HookProvider):
    def register_hooks(self, registry: HookRegistry) -> None:
        # TODO: register BeforeModelCallEvent and AfterModelCallEvent callbacks
        ...

# agent = Agent(model=make_model(), tools=[calculator], hooks=[TimingHook()])
# agent("What is 3 to the power of 5?")